In [4]:
pip install anthropic

Note: you may need to restart the kernel to use updated packages.


In [5]:
import anthropic
import pandas as pd

# Load the data
df = pd.read_csv(r'C:\Users\vaish\OneDrive\Desktop\Railway\Railway_raw_data.csv')
df['average_delay_minutes'] = df['average_delay_minutes'].fillna(0)

print("Data loaded successfully!")
print(f"Total records: {len(df)}")
print(f"Total trains: {df['train_number'].nunique()}")

Data loaded successfully!
Total records: 1829
Total trains: 90


In [6]:
# Prepare data summary for the chatbot
def get_data_summary():
    most_delayed = df.groupby('train_name')['average_delay_minutes'].mean().sort_values(ascending=False).head(5)
    most_punctual = df.groupby('train_name')['average_delay_minutes'].mean().sort_values().head(5)
    worst_stations = df.groupby('station_name')['average_delay_minutes'].mean().sort_values(ascending=False).head(5)
    
    summary = f"""
    Indian Railways Delay Dataset Summary:
    - Total records: {len(df)}
    - Total trains: {df['train_number'].nunique()}
    - Total unique stations: {df['station_name'].nunique()}
    - Overall average delay: {df['average_delay_minutes'].mean():.2f} minutes
    
    Top 5 Most Delayed Trains:
    {most_delayed.to_string()}
    
    Top 5 Most Punctual Trains:
    {most_punctual.to_string()}
    
    Top 5 Most Delayed Stations:
    {worst_stations.to_string()}
    """
    return summary

print(get_data_summary())


    Indian Railways Delay Dataset Summary:
    - Total records: 1829
    - Total trains: 90
    - Total unique stations: 480
    - Overall average delay: 33.37 minutes
    
    Top 5 Most Delayed Trains:
    train_name
Howrah Duronto          304.062500
Guwahati Sbc Express    129.219512
Howrah Mail             102.281690
Sbc Guwahati Express     83.575000
Kerala Express           59.964706
    
    Top 5 Most Punctual Trains:
    train_name
Gatimaan Express           4.875000
Swarna Jayanti Rajdhani    6.700000
Kolkata Rajdhani           6.777778
Rajdhani                   7.550000
Vande Bharat               8.107143
    
    Top 5 Most Delayed Stations:
    station_name
AKOLA JN        429.5
TATANAGAR JN    338.5
BILASPUR JN     299.5
DANKUNI         149.0
RAMPUR HAT      132.0
    


In [ ]:
# Build the chatbot
client = anthropic.Anthropic(api_key="YOUR_API_KEY")

def chat_with_railway_bot(user_question):
    data_summary = get_data_summary()
    
    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        messages=[
            {
                "role": "user",
                "content": f"""You are an Indian Railways delay analysis expert. 
                Answer questions based on this dataset:
                
                {data_summary}
                
                User question: {user_question}
                
                Give a clear, concise answer based on the data provided."""
            }
        ]
    )
    return message.content[0].text

# Test the chatbot
print(chat_with_railway_bot("Which train has the highest delay?"))

In [8]:
def railway_chatbot(question):
    question = question.lower()
    
    # Most delayed train
    if 'most delayed' in question or 'highest delay' in question or 'worst train' in question:
        train = df.groupby('train_name')['average_delay_minutes'].mean().idxmax()
        delay = df.groupby('train_name')['average_delay_minutes'].mean().max()
        return f"The most delayed train is {train} with an average delay of {delay:.2f} minutes."
    
    # Most punctual train
    elif 'punctual' in question or 'on time' in question or 'best train' in question:
        train = df.groupby('train_name')['average_delay_minutes'].mean().idxmin()
        delay = df.groupby('train_name')['average_delay_minutes'].mean().min()
        return f"The most punctual train is {train} with an average delay of only {delay:.2f} minutes."
    
    # Worst station
    elif 'worst station' in question or 'most delayed station' in question:
        station = df.groupby('station_name')['average_delay_minutes'].mean().idxmax()
        delay = df.groupby('station_name')['average_delay_minutes'].mean().max()
        return f"The most delayed station is {station} with an average delay of {delay:.2f} minutes."
    
    # Average delay
    elif 'average delay' in question or 'overall delay' in question:
        avg = df['average_delay_minutes'].mean()
        return f"The overall average delay across all trains is {avg:.2f} minutes."
    
    # Total trains
    elif 'how many trains' in question or 'total trains' in question:
        total = df['train_number'].nunique()
        return f"The dataset covers {total} unique trains."
    
    # Total stations
    elif 'how many stations' in question or 'total stations' in question:
        total = df['station_name'].nunique()
        return f"The dataset covers {total} unique stations."
    
    # Specific train delay
    elif 'tamil nadu' in question:
        delay = df[df['train_name'].str.contains('Tamil Nadu', na=False)]['average_delay_minutes'].mean()
        return f"Tamil Nadu Express has an average delay of {delay:.2f} minutes."
    
    elif 'rajdhani' in question:
        delay = df[df['train_name'].str.contains('Rajdhani', na=False)]['average_delay_minutes'].mean()
        return f"Rajdhani trains have an average delay of {delay:.2f} minutes."
    
    elif 'vande bharat' in question:
        delay = df[df['train_name'].str.contains('Vande Bharat', na=False)]['average_delay_minutes'].mean()
        return f"Vande Bharat trains have an average delay of {delay:.2f} minutes."
    
    elif 'shatabdi' in question:
        delay = df[df['train_name'].str.contains('Shatabdi', na=False)]['average_delay_minutes'].mean()
        return f"Shatabdi trains have an average delay of {delay:.2f} minutes."
    
    else:
        return "I can answer questions about: most delayed train, most punctual train, worst station, average delay, total trains, total stations, and specific trains like Rajdhani, Vande Bharat, Shatabdi, Tamil Nadu Express."

# Test it
print(railway_chatbot("Which train has the highest delay?"))
print(railway_chatbot("Which is the most punctual train?"))
print(railway_chatbot("What is the average delay?"))

The most delayed train is Howrah Duronto with an average delay of 304.06 minutes.
The most punctual train is Gatimaan Express with an average delay of only 4.88 minutes.
The overall average delay across all trains is 33.37 minutes.


In [ ]:
# Interactive chatbot loop
print("🚂 Indian Railways Delay Analysis Chatbot")
print("=" * 50)
print("Ask me anything about Indian Railways delays!")
print("Type 'quit' to exit\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == 'quit':
        print("Goodbye!")
        break
    response = railway_chatbot(user_input)
    print(f"Bot: {response}\n")

🚂 Indian Railways Delay Analysis Chatbot
Ask me anything about Indian Railways delays!
Type 'quit' to exit



You:  "Which train has the highest delay?"


Bot: The most delayed train is Howrah Duronto with an average delay of 304.06 minutes.



You:  "How many stations are there?"


Bot: The dataset covers 480 unique stations.



You:  "Tell me about Vande Bharat"


Bot: Vande Bharat trains have an average delay of 8.97 minutes.

